# 04 · Evaluation

| Stage | Notebook |
|---|---|
| 1. Data quality & EDA | `01_eda.ipynb` |
| 2. Feature engineering | `02_feature_engineering.ipynb` |
| 3. Model training | `03_model_training.ipynb` |
| **4. Evaluation** | **this notebook** |

**Inputs**, saved by `scripts/run_phase1.py`:
* `results/predictions.csv.gz` contains raw scores for the labelled modelling rows, outcomes, and outcome-availability dates. Calibration, metrics, policies, monthly means and bootstrap calculations are recomputed here and checked against saved tables.
* Ablation, importance, target sensitivity, PSI and adversarial-validation tables are loaded from `results/`. They involve additional features or model refits in the pipeline.

**Part A** records current selection evidence from development CV / Sep–Oct. **Part B** reports test findings. The test period was inspected during development (§0), so Part B is exploratory. Table assertions do not validate the surrounding Markdown interpretations.

Reference rows are the prior and route-history heuristic; learned rows identify their calibration variants.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent / "src"))  # make `aerocast` importable
import warnings; warnings.filterwarnings("ignore")
import json
import numpy as np, pandas as pd
from IPython.display import display
from aerocast import artefacts, config, evaluate, pipeline, plots
pd.set_option("display.precision", 4); pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 100)
pd.set_option("display.max_columns", 40)
R = config.RESULTS_DIR
S = json.loads((R / "summary.json").read_text())
FINAL = config.FINAL_MODEL

labels, raw_preds = pipeline.load_predictions()
print("final model (fixed before test):", S["final_model"]["variant"], "| model family:", S["final_model"]["model"])

final model (fixed before test): LogReg (Platt, rule-selected) | model family: logreg


<a id="protocol"></a>
## 0. Development protocol (disclosure)

The Nov–Dec window was reserved for final evaluation but was inspected repeatedly during development. This records that exposure and the current selection evidence; it does not restore an untouched test set.

**Diagnostics that printed test metrics before the design was frozen:**
1. **First run** (single-September tuning design): test PR-AUC and calibration of all models were printed. The design was abandoned because of the September evidence (XGBoost stopped at 13 trees), but test numbers had been seen.
2. **Month features:** a with/without comparison printed test PR-AUC, and that was the original justification for removing them. **Withdrawn.** Re-justified by rolling CV (notebook 02, §7).
3. **Feature-subset variants** (dropping log-counts, IDs, day-level history, congestion; a minimal set) printed calibration-window *and* test PR-AUC and test recall@precision. These shaped the "day-level features memorise days" hypothesis. **Re-justified** by the distinct-value analysis and the CV ablation (notebook 02, §8; Part A.4 below).
4. **Precision target:** 0.50 → 0.40 was decided because 0.50 is unreachable on Sep–Oct (base rate 15.7%), but the same diagnostic also printed test recall at 0.35/0.40/0.50.
5. **Calibration:** test ECE of raw vs Platt vs isotonic was seen *before* any selection rule existed. The "fit Sep → check Oct" rule, the shift-aware Platt variant (evaluated on test ECE), and the 0.005-margin rule were all introduced **after** that.
6. **First review round:** the full pipeline (which also computes test metrics) ran once while the final-model comparison still used XGBoost's optimistic in-fold early-stopping score. The comparison was corrected for a methodological reason identified from CV alone (notebook 03, §7). That correction reversed the choice to LogReg.
7. **Reference baselines in the policy table:** adding the *prior* to the policy comparison showed an apparent lift on both Sep–Oct and test for a constant score. The cause was an order-based tie-break in the daily top-k% policy (rows are in departure order, so ties picked the day's latest flights). The tie-break is now seeded random. This is an evaluation-hygiene fix. It leaves model scores, the policy definition and the share (10%) unchanged, and it only alters the daily-policy numbers of scores that contain exact ties (the prior, and route history to a small extent).
8. **Earlier second review round (items 1–11):** labels, baselines, assertions and documentation only. No modelling decision changed. The joint re-tuning of (m, window, C) re-selected the same values (Part A.2).

**Current selection evidence (retain the exposure disclosure above):** the decision clock (and clock vs 2-day lag, CV), smoothing m and window (CV, jointly with C), C and XGBoost configuration (CV), final model (CV as deployed), calibration margin rule and its outcome (Sep→Oct), rolling-recalibration window N = 28 (a priori), daily top-10% policy (a priori capacity assumption), and the precision target (Sep–Oct).

**Implication.** Test numbers are not perfectly clean confirmations. The design was iterated with test results visible on several occasions, so treat small test differences with caution. Every conclusion below is stated with its CV or Sep–Oct basis, and Part B is labelled exploratory for that reason.

**Fitting-clock correction in the saved run:** final fitting excludes 167 unavailable training outcomes (211,517 labels used); static calibration/threshold selection excludes 251 unavailable calibration outcomes (55,377 used). CV fitting and scoring for selection also enforce availability deadlines. Scoring cohorts retain these late outcomes for retrospective checks once they are known. Configuration selection and reported development CV use the same folds, so the scores are not independent nested-CV estimates.

## 1. Metric theory in brief

* **PR-AUC (average precision).** sklearn's `average_precision_score` computes $AP = \sum_k (R_k - R_{k-1})\,P_k$, a step-wise sum over thresholds. It differs from trapezoidal integration of the PR curve, which uses linear interpolation. "PR-AUC" in the tables means AP. The constant-prior reference has AP equal to the evaluated base rate.
* **PR vs ROC under imbalance.** ROC-AUC is the probability that a random late flight is ranked above a random on-time flight. It ignores the base rate, so the same ROC can mean very different precision when positives are rare. PR curves put precision on the axis, which is what an operations desk experiences. Both are reported.
* **ECE** (Guo et al., 2017): $\mathrm{ECE} = \sum_b \frac{|B_b|}{n}\,\big|\bar y_b - \bar p_b\big|$ over bins of predicted probability.
  * Finite-sample noise and bin choice affect ECE: a calibrated model can have positive measured ECE, and using more bins can increase sampling error. A single ECE value is not a certificate of calibration.
  * Results also depend on the binning. **Equal-width** bins (15) leave the sparse high-risk tail with few flights; **approximately equal-mass** bins (`ece_equal_mass`) start with 15 balanced rank bins, move each boundary to the end of its tied-probability group, and collapse duplicate boundaries. Identical probabilities stay together regardless of row order; ties can produce fewer bins or unequal counts. Constant predictions use one bin. Both are reported, and the Brier score (proper, binning-free) is reported alongside.
  * ECE is a **marginal** check: the *prior* (a constant equal to the training base rate) has a small ECE whenever the test base rate happens to be near the training one. A low ECE therefore says nothing about ranking skill, which is why ECE is always read together with PR-AUC and Brier.
* **Platt / temperature scaling vs isotonic.** Platt fits $\sigma(a\,\mathrm{logit}(p)+b)$: 2 parameters, monotone, smooth, data-efficient. Temperature scaling is its 1-parameter special case ($b=0$), the multi-class tool in Guo et al. Isotonic regression fits any monotone step function: more flexible, but needs more data and can overfit. All of them assume the calibration data are **distributed like deployment data**.
* **`oracle_recall@p`.** The best recall at precision ≥ p, with the threshold **tuned on the evaluated labels themselves**. It is a model-comparison diagnostic; it does not evaluate a threshold frozen before those outcomes were observed. Part B separately evaluates the pre-specified policies.
* **Why the bootstrap resamples whole days.** Flights can share daily operating conditions, so resampling individual flights can understate uncertainty. Resampling whole days preserves within-day dependence; it does not preserve dependence between adjacent days. Model differences use the same resampled days (paired).

# Part A · Evidence used for selection (CV / Sep–Oct)

*Everything below uses the training window (rolling-origin CV) or the calibration window (Sep–Oct). No test labels. Linked to the Development protocol: [§0](#protocol).*

## A.1 Final-model selection: rolling-CV PR-AUC, as deployed

In [2]:
cv_tbl = pd.read_csv(R / "cv_summary.csv")
cv_tbl.drop(columns="variant").round(4)

,model,calibration,cv_pr_auc_mean,cv_pr_auc_std,fold_m5,fold_m6,fold_m7,fold_m8
0,Prior (base rate),none,0.2777,0.0481,0.2267,0.3152,0.3221,0.2467
1,Route history,none,0.3477,0.0601,0.2766,0.3911,0.4034,0.3197
2,LogReg (as deployed),raw,0.4714,0.0711,0.4343,0.5052,0.5525,0.3936
3,"XGBoost (as deployed, fixed trees)",raw,0.4500,0.0671,0.4315,0.4253,0.5479,0.3953
4,"XGBoost (in-fold early stopping, optimistic; not used)",raw,0.4749,0.0693,0.4485,0.4990,0.5570,0.3950


In [3]:
pair = S["cv_paired_logreg_minus_xgboost"]
pd.Series({"per-fold LogReg − XGBoost (May..Aug)": np.round(pair["diffs"], 4).tolist(), "mean": pair["mean"],
           "sd": pair["std"], "folds LogReg better": f'{pair["folds_a_better"]}/{pair["n_folds"]}'}, name="paired").to_frame()

,paired
per-fold LogReg − XGBoost (May..Aug),"[0.0027, 0.08, 0.0046, -0.0016]"
mean,0.0214
sd,0.0391
folds LogReg better,3/4


LogReg has the higher development CV mean (0.4714 versus 0.4500), so it is selected by the declared rule. The test AP-difference interval (XGBoost − LogReg, [-0.010, +0.040]) includes zero: it does not establish a test-performance difference, equivalence, or noninferiority. Test exposure during development remains disclosed. Reference CV AP: route history **0.3477**, prior **0.2777**.

## A.2 Feature-design evidence (CV): history settings, decision clock, month

Each history pair is evaluated at its best C on the development folds. The clock rows repeat the configured m = 5 / 14-day pair at 00:00 and 24:00 on D-1, moving fitting/selection and feature-history cutoffs together.

In [4]:
tuning = pd.read_csv(R / "history_tuning.csv")
tuning.round(4)

,smoothing_m,long_window,decision_hour,best_C,fold_m5,fold_m6,fold_m7,fold_m8,cv_pr_auc_mean,cv_pr_auc_std,cv_pr_auc_mean_C0.001
0,5,14,20,0.100,0.4343,0.5052,0.5525,0.3936,0.4714,0.0711,0.4692
1,5,28,20,0.001,0.4385,0.4967,0.5405,0.3940,0.4674,0.0643,0.4674
2,5,56,20,0.010,0.4344,0.5035,0.5456,0.3916,0.4688,0.0689,0.4671
3,20,14,20,0.010,0.4300,0.5055,0.5506,0.3935,0.4699,0.0712,0.4678
4,20,28,20,0.001,0.4362,0.4959,0.5397,0.3944,0.4666,0.0641,0.4666
5,20,56,20,0.010,0.4316,0.5028,0.5445,0.3914,0.4676,0.0689,0.4658
6,50,14,20,0.010,0.4229,0.5022,0.5479,0.3901,0.4658,0.0722,0.4642
7,50,28,20,0.001,0.4322,0.4931,0.5378,0.3932,0.4641,0.0640,0.4641
8,50,56,20,0.003,0.4288,0.5006,0.5408,0.3903,0.4652,0.0681,0.4636
9,5,14,0,0.003,0.4063,0.5018,0.5411,0.3868,0.4590,0.0743,0.4584


In [5]:
grid = tuning[tuning.decision_hour == config.DECISION_HOUR].pivot(index="smoothing_m", columns="long_window", values="cv_pr_auc_mean")
grid.round(4)

long_window,14,28,56
smoothing_m,,,
5,0.4714,0.4674,0.4688
20,0.4699,0.4666,0.4676
50,0.4658,0.4641,0.4652


* The selected settings remain **m = 5, long window = 14 days, C = 0.1**, mean CV AP **0.4714**; fixed-C = 0.001 AP is **0.4692**.
* The 20:00 clock scores 0.4714, compared with 0.4590 at 00:00 (approximately a two-day lag) and 0.4812 at 24:00. The 24:00 scenario permits outcomes unavailable at the chosen 20:00 deadline. These scenarios move fitting and selection cutoffs as well as feature-history cutoffs, so their score differences do not isolate a feature-leakage effect.

In [6]:
mcv = pd.read_csv(R / "month_feature_cv.csv")
mcv.round(4)

,features,model,fold_m5,fold_m6,fold_m7,fold_m8,cv_pr_auc_mean
0,without month,logreg,0.4343,0.5052,0.5525,0.3936,0.4714
1,without month,xgboost,0.4315,0.4253,0.5479,0.3953,0.4500
2,with month,logreg,0.4250,0.5066,0.5491,0.3955,0.4691
3,with month,xgboost,0.4295,0.4564,0.5476,0.3858,0.4548


Adding cyclic month features lowers LogReg mean CV AP from 0.4714 to 0.4691 (two folds improve, two worsen). For fixed-tree XGBoost the mean rises from 0.4500 to 0.4548, but three of four folds worsen; the gain comes from June alone (+0.031). Training contains only months 1–8 in one year, so Sep–Dec month values never occur in training; these checks do not establish a reusable seasonal effect. **Decision: no month feature**, supported by the final model's development CV. The earlier test-based justification remains withdrawn.

## A.3 Calibration rule, decided on Sep–Oct

Fit Platt/isotonic on September labels available before October predictions; check fixed-width ECE on October labels available before November predictions. Keep raw scores unless the best recalibrator improves ECE by more than **0.005**. Refit the selected static variant on **55,377 eligible Sep–Oct outcomes**, excluding 251 unavailable labels. The earlier lowest-ECE rule and development-time test exposure remain disclosed in §0.

In [7]:
final_preds, choice, calib_tbl = pipeline.calibrate(labels, raw_preds)
print(artefacts.assert_matches(calib_tbl, "calibrator_selection.csv"))
print("rule-selected variants:", {m: pipeline.variant_label(f"{m}_rule_selected", choice)[0] for m in pipeline.MODELS})
calib_tbl.set_index(["model", "calibrator"]).round(4)

OK: identical to results/calibrator_selection.csv
rule-selected variants: {'logreg': 'LogReg (Platt, rule-selected)', 'xgboost': 'XGBoost (raw, rule-selected)'}


check_ece  check_ece_equal_mass  check_brier  gain_vs_raw
model   calibrator                                                           
logreg  raw            0.0626                0.0633       0.1399       0.0000
        platt          0.0329                0.0329       0.1368       0.0297
        isotonic       0.0339                0.0336       0.1376       0.0287
xgboost raw            0.0235                0.0238       0.1379       0.0000
        platt          0.0432                0.0432       0.1401      -0.0197
        isotonic       0.0427                0.0427       0.1404      -0.0193

* **LogReg → Platt:** October fixed-width ECE improves from **0.0626 to 0.0329**, a gain of **0.0297**, exceeding the 0.005 margin.
* **XGBoost → raw:** neither recalibrator improves on raw by the margin.
* **Rolling Platt**, with a pre-specified 28-day window, is a separate adaptive comparison. The 14/56-day windows are sensitivity analyses; test results do not select a replacement final model.

## A.4 Feature-family ablation: the CV evidence

In [8]:
steps = pd.read_csv(R / "ablation_steps.csv")
steps[["model", "calibration", "step", "cv_gain_mean", "cv_folds_improved"]].round(4)

,model,calibration,step,cv_gain_mean,cv_folds_improved
0,LogReg,raw,+ congestion,0.0205,4/4
1,LogReg,raw,+ history_entity,0.0201,4/4
2,LogReg,raw,+ history_day,0.0114,3/4
3,XGBoost,raw,+ congestion,0.0085,3/4
4,XGBoost,raw,+ history_entity,0.0068,2/4
5,XGBoost,raw,+ history_day,-0.0015,3/4


At the selected hyperparameters, LogReg mean CV AP increases with congestion (+0.0205, 4/4 folds), entity history (+0.0201, 4/4), and day-level history (+0.0114, 3/4). For XGBoost, the corresponding mean gains are +0.0085 (3/4), +0.0068 (2/4), and −0.0015 (3/4). The last result has a slightly negative mean despite improvement in three folds. These are development comparisons at the chosen settings, not independent estimates after nested selection.

The full table includes prior and route-history reference rows on the same folds.

In [9]:
abl = pd.read_csv(R / "ablation.csv")
abl[["model", "calibration", "features", "n_features", "cv_pr_auc_mean", "cv_pr_auc_std"]].round(4)

,model,calibration,features,n_features,cv_pr_auc_mean,cv_pr_auc_std
0,Prior (base rate),none,(baseline),0,0.2777,0.0481
1,Route history,none (smoothed rate),(baseline),0,0.3477,0.0601
2,LogReg,raw,schedule + ids,13,0.4193,0.0634
3,LogReg,raw,+ congestion,18,0.4399,0.0717
4,LogReg,raw,+ history_entity,36,0.4600,0.0730
5,LogReg,raw,+ history_day,43,0.4714,0.0711
6,LogReg,raw,history only,25,0.4480,0.0760
7,XGBoost,raw,schedule + ids,13,0.4363,0.0749
8,XGBoost,raw,+ congestion,18,0.4448,0.0844
9,XGBoost,raw,+ history_entity,36,0.4515,0.0829


In [10]:
figs = {}
figs["ablation_cv"] = plots.ablation_bars(abl); plots.save(figs["ablation_cv"], "ablation_cv"); figs["ablation_cv"]

<Figure size 650x370 with 1 Axes>

## A.5 Grouped permutation importance (Sep–Oct, not test)

Each family is shuffled **together** with one permutation (5 repeats). Correlated features within a family can't stand in for each other, and the history features *are* strongly correlated (notebook 01, §8). This is descriptive, not a selection criterion.

In [11]:
perm = pd.read_csv(R / "permutation_importance_groups.csv")
display(perm.pivot(index="group", columns="model", values="pr_auc_drop").round(4))
gain = pd.read_csv(R / "xgb_gain_importance.csv", index_col=0)["gain"]
psi = pd.read_csv(R / "psi.csv")
figs.update(pipeline.evaluation_figures(labels, final_preds, choice, pipeline.monthly_drift(labels, final_preds, choice), gain, perm, abl, psi))
figs["perm_importance_groups"]

model,LogReg,XGBoost
group,,
categorical,0.0381,0.0133
congestion,0.0458,0.0117
history_day,0.0263,0.0032
history_entity,0.0290,0.0190
schedule,0.0482,0.0501


<Figure size 550x220 with 1 Axes>

Grouped permutation gives LogReg AP drops of schedule 0.048, congestion 0.046, categorical 0.038, entity history 0.029, and day-level history 0.026. For XGBoost: schedule 0.050, entity history 0.019, categorical 0.013, congestion 0.012, and day-level history 0.003. These describe sensitivity to shuffled feature groups; they do not identify causal effects. For XGBoost, the ordering by drop is schedule, entity history, categorical, congestion, then day-level history.

LogReg coefficients are conditional on correlated features. Opposite signs on overlapping history-volume features should not be read as independent operational effects.

In [12]:
pd.read_csv(R / "logreg_coefficients.csv", index_col=0).head(10)

,coef
num__hist_logn_carrier_14d,0.6773
num__hist_logn_carrier_7d,-0.6500
cat__dest_BOS,-0.5850
cat__dest_PHL,0.4861
cat__origin_EWR,-0.4469
cat__dest_LAX,-0.4112
cat__dest_SNA,-0.3862
cat__dest_ORD,-0.3327
cat__carrier_DL,-0.3177
cat__dest_PSE,-0.3174


# Part B · Exploratory test-set findings

Test = Nov–Dec 2013. The current rules use Part A's development evidence, but test numbers were visible during development (§0). Re-justifying rules on CV does not restore an untouched test set. Model differences and diagnostic interpretations are exploratory.

**B.1** evaluates the current rules; **B.2** contains calibration, ablation, target and shift diagnostics.

## B.1 Pre-specified rules evaluated on test

### B.1.1 Headline test metrics: baselines, calibration variants and the final model

In [13]:
thresholds = pipeline.frozen_thresholds(labels, final_preds)
results = pipeline.metrics_table(labels, final_preds, thresholds, choice)
print(artefacts.assert_matches(results, "metrics_all.csv"))
headline = pipeline.headline_table(results)
print(artefacts.assert_matches(headline, "headline_test.csv"))
headline[["model", "calibration", *pipeline.SHOW_COLS]]

OK: identical to results/metrics_all.csv
OK: identical to results/headline_test.csv


,model,calibration,pr_auc,roc_auc,brier,ece,ece_equal_mass,mean_pred,base_rate,oracle_recall@p0.30,oracle_recall@p0.40,oracle_recall@p0.50
0,Prior (base rate),none,0.2583,0.5000,0.1916,0.0067,0.0067,0.2650,0.2583,0.0000,0.0000,0.0000
1,Route history,none (smoothed rate),0.3534,0.6208,0.1869,0.0373,0.0380,0.2420,0.2583,0.7868,0.2444,0.0163
2,LogReg (raw),raw,0.3934,0.6532,0.1812,0.0187,0.0224,0.2584,0.2583,0.8454,0.4206,0.1276
3,"LogReg (Platt, rule-selected)",Platt (rule-selected),0.3934,0.6532,0.1874,0.0824,0.0824,0.1759,0.2583,0.8454,0.4206,0.1276
4,LogReg (rolling Platt 28d),rolling Platt 28d (causal),0.3916,0.6570,0.1810,0.0271,0.0267,0.2316,0.2583,0.8697,0.4282,0.1186
5,"XGBoost (raw, rule-selected)",raw (rule-selected),0.4074,0.6655,0.1793,0.0269,0.0269,0.2314,0.2583,0.8698,0.4833,0.1699
6,XGBoost (rolling Platt 28d),rolling Platt 28d (causal),0.3964,0.6635,0.1798,0.0233,0.0218,0.2446,0.2583,0.8777,0.4376,0.1293


The table separates reference rows and learned calibration variants. Static Platt preserves LogReg ranking in this run, so its AP equals the raw model's **0.393**; its test ECE is **0.082**, versus **0.019** raw and **0.027** rolling Platt.

Raw XGBoost AP is **0.407**, route history **0.353**, and prior **0.258**. These ranking metrics do not prove equivalence or explain why one family performs differently.

The prior's fixed-width and corrected equal-mass ECE are both 0.007. All its probabilities are identical, so equal-mass ECE uses one bin and equals the absolute gap between mean prediction and observed base rate. This reflects close overall means, not ranking skill. `oracle_recall@p` chooses a threshold on the evaluated labels and is a comparison metric only.

### B.1.2 November and December separately

In [14]:
by_month = pipeline.headline_by_month(results)
print(artefacts.assert_matches(by_month, "headline_by_month.csv"))
by_month[["model", "calibration", "split", "base_rate", "pr_auc", "roc_auc", "brier", "ece", "mean_pred"]]

OK: identical to results/headline_by_month.csv


,model,calibration,split,base_rate,pr_auc,roc_auc,brier,ece,mean_pred
0,Prior (base rate),none,test_m11,0.1810,0.1810,0.5000,0.1553,0.0840,0.2650
1,Route history,none (smoothed rate),test_m11,0.1810,0.2208,0.5596,0.1492,0.0352,0.1750
2,LogReg (raw),raw,test_m11,0.1810,0.2909,0.6546,0.1456,0.0560,0.2355
3,"LogReg (Platt, rule-selected)",Platt (rule-selected),test_m11,0.1810,0.2909,0.6546,0.1418,0.0218,0.1596
4,LogReg (rolling Platt 28d),rolling Platt 28d (causal),test_m11,0.1810,0.2848,0.6494,0.1419,0.0051,0.1773
5,"XGBoost (raw, rule-selected)",raw (rule-selected),test_m11,0.1810,0.2726,0.6382,0.1433,0.0178,0.1934
6,XGBoost (rolling Platt 28d),rolling Platt 28d (causal),test_m11,0.1810,0.2629,0.6311,0.1437,0.0129,0.1753
7,Prior (base rate),none,test_m12,0.3355,0.3355,0.5000,0.2279,0.0704,0.2650
8,Route history,none (smoothed rate),test_m12,0.3355,0.3944,0.5740,0.2246,0.0602,0.3088
9,LogReg (raw),raw,test_m12,0.3355,0.4556,0.6313,0.2168,0.0600,0.2813


Static Platt has LogReg ECE 0.022 in November and 0.143 in December (December base rate 33.5%): **the static rule fails in December**. Raw LogReg has test ECE 0.019 overall, but its monthly errors are larger; a close overall mean prediction and base rate do not establish reliable probabilities in every month. Rolling Platt with the pre-specified 28-day window has test ECE 0.027 for LogReg (November 0.005, December 0.050) and 0.023 for XGBoost. LogReg sensitivity windows of 14 and 56 days give test ECE 0.021 and 0.051. These test diagnostics do not select a new final variant or prove a cause of the errors.

### B.1.3 LogReg vs XGBoost: day-block bootstrap (raw-ranking metrics, rule-selected variants)

In [15]:
boot = pipeline.bootstrap_compare(labels, final_preds)
assert json.loads(json.dumps(boot)) == S["bootstrap_test"], "bootstrap differs from summary.json"
pd.DataFrame({k: {"XGBoost 95% CI": np.round(v["xgboost_ci95"], 3), "LogReg 95% CI": np.round(v["logreg_ci95"], 3),
                  "XGB − LR 95% CI": np.round(v["diff_xgboost_minus_logreg_ci95"], 3),
                  "P(XGB not better)": v["p_xgboost_not_better"]} for k, v in boot.items()}).T

,XGBoost 95% CI,LogReg 95% CI,XGB − LR 95% CI,P(XGB not better)
pr_auc,"[0.346, 0.468]","[0.34, 0.45]","[-0.01, 0.04]",0.152
roc_auc,"[0.637, 0.694]","[0.62, 0.682]","[-0.008, 0.032]",0.116


XGBoost's test AP exceeds LogReg's by **0.0140**, with a paired 95% interval **[−0.010, +0.040]**. Including zero does not demonstrate equivalence, noninferiority, or a clear difference. The final family remains LogReg under the development CV rule.

These experiments do not show that manual summaries exhaust the useful interactions. Learned embeddings and properly ordered temporal representations are hypotheses for later-phase comparisons.

### B.1.4 PR / ROC curves and reliability (references included)

In [16]:
for name, fig in figs.items():
    plots.save(fig, name)
figs["pr_roc_test"]

<Figure size 850x360 with 2 Axes>

In [17]:
figs["reliability_logreg_test"]

<Figure size 440x480 with 2 Axes>

In [18]:
figs["reliability_xgboost_test"]

<Figure size 440x480 with 2 Axes>

The dashed grey lines are the references: the prior (horizontal base-rate line in PR, the diagonal in ROC) and, in the reliability plots, the route-history heuristic. The reliability plots show the raw, static-Platt and rolling-Platt variants of each model.

### B.1.5 Operating policies

* **Primary:** flag the highest-risk 10% each day under a pre-specified review-share assumption. A positive-slope monotone mapping within a day preserves ordering, but the resulting precision and recall can still change under distribution shift. Exact ties use the existing seeded random rule.
* **Secondary:** freeze a probability threshold selected from eligible Sep–Oct outcomes to target precision ≥ 0.40. That empirical target is not a guarantee on new periods. The earlier 0.50 → 0.40 change and associated test exposure are disclosed in §0.

In [19]:
policies = pipeline.operating_policies(labels, final_preds, thresholds, choice)
print(artefacts.assert_matches(policies, "operating_policies.csv"))
policies[["model", "calibration", "split", "role", "policy", "flag_rate", "precision", "recall", "lift",
          "f1", "accuracy", "tp", "fp", "fn", "tn", "threshold"]].round(3)

OK: identical to results/operating_policies.csv


,model,calibration,split,role,policy,flag_rate,precision,recall,lift,f1,accuracy,tp,fp,fn,tn,threshold
0,Prior (base rate),none,calib,primary,daily top 10%,0.100,0.153,0.098,0.977,0.120,0.773,857,4732,7874,42165,NaN
1,Prior (base rate),none,calib,secondary,frozen thr for P>=0.4 (from Sep-Oct),0.000,NaN,0.000,NaN,0.000,0.843,0,0,8731,46897,inf
2,Prior (base rate),none,test,primary,daily top 10%,0.101,0.243,0.095,0.941,0.136,0.690,1319,4109,12627,35936,NaN
3,Prior (base rate),none,test,secondary,frozen thr for P>=0.4 (from Sep-Oct),0.000,NaN,0.000,NaN,0.000,0.742,0,0,13946,40045,inf
4,Route history,none (smoothed rate),calib,primary,daily top 10%,0.100,0.232,0.148,1.476,0.181,0.789,1295,4294,7436,42603,NaN
5,Route history,none (smoothed rate),calib,secondary,frozen thr for P>=0.4 (from Sep-Oct),0.001,0.417,0.002,2.655,0.005,0.843,20,28,8711,46869,0.469
6,Route history,none (smoothed rate),test,primary,daily top 10%,0.101,0.358,0.139,1.387,0.201,0.713,1944,3484,12002,36561,NaN
7,Route history,none (smoothed rate),test,secondary,frozen thr for P>=0.4 (from Sep-Oct),0.059,0.425,0.097,1.646,0.158,0.733,1350,1826,12596,38219,0.469
8,"LogReg (Platt, rule-selected)",Platt (rule-selected),calib,primary,daily top 10%,0.100,0.302,0.193,1.923,0.236,0.803,1687,3902,7044,42995,NaN
9,"LogReg (Platt, rule-selected)",Platt (rule-selected),calib,secondary,frozen thr for P>=0.4 (from Sep-Oct),0.038,0.401,0.096,2.552,0.155,0.836,838,1254,7893,45643,0.329


#### Accuracy, precision, recall, F1 and confusion matrix

These need a yes/no decision, so they are reported at the two operating points above, both fixed before test (no threshold is tuned on test labels).

* **Accuracy is misleading at this class balance.** Predicting every flight on time is already **74.2%** accurate on test (the prior's frozen-threshold row: it never reaches the precision target, so it flags nothing). LogReg scores **0.740** at the frozen threshold and **0.731** at the daily top 10%, no better than that trivial rule, yet it catches 1,972 and 2,430 late flights the trivial rule misses.
* **Frozen threshold (LogReg):** precision 0.491, recall 0.141, F1 0.220. **Daily top 10%:** precision 0.448, recall 0.174, F1 0.251.
* **Most errors are missed delays:** of 13,946 late test flights, 11,974 are predicted on time at the frozen threshold, against 2,041 false alarms.

In [20]:
figs["confusion_logreg_test"]

<Figure size 680x330 with 2 Axes>

### Decision support: rescheduling / add-buffer candidates

Each test day, the final model's ranking selects the top-k% flights as candidates for a capacity-limited review desk, k in {5, 10, 15, 20}, against the prior (random) and route-history rankings.

**This shows ranking quality, not realised operational savings**: the data contain no cost or schedule-change model.

In [21]:
resched = pipeline.rescheduling_candidates(labels, final_preds, choice)
print(artefacts.assert_matches(resched, "rescheduling_candidates.csv"))
resched[["model", "calibration", "top_share", "flagged_per_day", "precision", "recall", "lift", "delayed_per_100_flagged"]].round(3)

OK: identical to results/rescheduling_candidates.csv


,model,calibration,top_share,flagged_per_day,precision,recall,lift,delayed_per_100_flagged
0,Prior (base rate),none,0.05,44.705,0.251,0.049,0.971,25.083
1,Prior (base rate),none,0.10,88.984,0.243,0.095,0.941,24.300
2,Prior (base rate),none,0.15,133.148,0.250,0.145,0.967,24.982
3,Prior (base rate),none,0.20,177.393,0.252,0.195,0.974,25.164
4,Route history,none (smoothed rate),0.05,44.705,0.386,0.076,1.495,38.614
5,Route history,none (smoothed rate),0.10,88.984,0.358,0.139,1.387,35.814
6,Route history,none (smoothed rate),0.15,133.148,0.346,0.201,1.339,34.597
7,Route history,none (smoothed rate),0.20,177.393,0.339,0.263,1.313,33.916
8,"LogReg (Platt, rule-selected)",Platt (rule-selected),0.05,44.705,0.474,0.093,1.834,47.378
9,"LogReg (Platt, rule-selected)",Platt (rule-selected),0.10,88.984,0.448,0.174,1.733,44.768


In [22]:
# the primary policy only depends on within-day ranks: raw / Platt / rolling Platt flag (almost) the same flights
flag = {k: evaluate.flag_daily_top_share(labels.dates["test"], final_preds[k]["test"])
        for k in ["logreg_raw", "logreg_rule_selected", f"logreg_rolling{config.ROLLING_CALIB_DAYS}"]}
pd.Series({f"{a} vs {b}": float((flag[a] == flag[b]).mean()) for a, b in
           [("logreg_raw", "logreg_rule_selected"), ("logreg_raw", f"logreg_rolling{config.ROLLING_CALIB_DAYS}")]},
          name="share of flights flagged identically").to_frame()

,share of flights flagged identically
logreg_raw vs logreg_rule_selected,1.0
logreg_raw vs logreg_rolling28,1.0


**Primary result (LogReg, static Platt rule-selected, test): precision 0.448, recall 0.174, lift 1.73×.** The prior lift is 0.94, route history 1.39, and XGBoost 1.70. The LogReg threshold frozen on eligible Sep–Oct outcomes is 0.329078; it flags 3.8% of calibration flights and 7.4% of test flights. Test precision is 0.491 and recall 0.141. Ranking-based selection keeps the chosen daily share; its precision and recall can still change across periods. The prior threshold is infinite because it does not meet the calibration precision target, so it flags nothing.

## B.2 Exploratory (post-hoc) findings

### B.2.1 Calibration study: every variant on test, Nov and Dec

In [23]:
study = pipeline.calibration_study(results)
print(artefacts.assert_matches(study, "calibration_study.csv"))
study[["model", "calibration", "ece_test", "ece_test_m11", "ece_test_m12", "ece_equal_mass_test", "brier_test", "brier_test_m11", "brier_test_m12", "mean_pred_test"]].round(4)

OK: identical to results/calibration_study.csv


,model,calibration,ece_test,ece_test_m11,ece_test_m12,ece_equal_mass_test,brier_test,brier_test_m11,brier_test_m12,mean_pred_test
0,Prior (base rate),none,0.0067,0.0840,0.0704,0.0067,0.1916,0.1553,0.2279,0.2650
1,Route history,none (smoothed rate),0.0373,0.0352,0.0602,0.0380,0.1869,0.1492,0.2246,0.2420
2,LogReg (raw),raw,0.0187,0.0560,0.0600,0.0224,0.1812,0.1456,0.2168,0.2584
3,"LogReg (Platt, static)","Platt (static, fit Sep-Oct)",0.0824,0.0218,0.1434,0.0824,0.1874,0.1418,0.2330,0.1759
4,"LogReg (isotonic, static)","isotonic (static, fit Sep-Oct)",0.0823,0.0234,0.1423,0.0810,0.1875,0.1420,0.2329,0.1773
5,LogReg (rolling Platt 14d),rolling Platt 14d (causal),0.0214,0.0111,0.0403,0.0221,0.1807,0.1432,0.2182,0.2511
6,LogReg (rolling Platt 28d),rolling Platt 28d (causal),0.0271,0.0051,0.0503,0.0267,0.1810,0.1419,0.2199,0.2316
7,LogReg (rolling Platt 56d),rolling Platt 56d (causal),0.0512,0.0134,0.0890,0.0512,0.1821,0.1417,0.2224,0.2071
8,XGBoost (raw),raw,0.0269,0.0178,0.0661,0.0269,0.1793,0.1433,0.2152,0.2314
9,"XGBoost (Platt, static)","Platt (static, fit Sep-Oct)",0.0803,0.0319,0.1298,0.0803,0.1860,0.1439,0.2280,0.1781


Static Platt has LogReg ECE 0.022 in November and 0.143 in December (December base rate 33.5%): **the static rule fails in December**. Raw LogReg has test ECE 0.019 overall, but its monthly errors are larger; a close overall mean prediction and base rate do not establish reliable probabilities in every month. Rolling Platt with the pre-specified 28-day window has test ECE 0.027 for LogReg (November 0.005, December 0.050) and 0.023 for XGBoost. LogReg sensitivity windows of 14 and 56 days give test ECE 0.021 and 0.051. These test diagnostics do not select a new final variant or prove a cause of the errors.

Recent-outcome recalibration is a possible adaptation strategy, not a universal remedy. The experiments do not isolate pure prior shift, establish causal mechanisms, or justify choosing a new final variant from the already-seen test results.

### B.2.2 Monthly calibration drift

In [24]:
drift = pipeline.monthly_drift(labels, final_preds, choice)
print(artefacts.assert_matches(drift, "monthly_calibration_drift.csv"))
figs["monthly_drift_final"]

OK: identical to results/monthly_calibration_drift.csv


<Figure size 650x320 with 1 Axes>

In [25]:
drift.pivot_table(index="month", columns="model", values=["observed", "predicted"]).round(3)

observed                                                                    predicted                                                      
model LogReg (Platt, static) LogReg (raw) LogReg (rolling Platt 28d) XGBoost (raw) LogReg (Platt, static) LogReg (raw) LogReg (rolling Platt 28d) XGBoost (raw)
month                                                                                                                                                          
1                      0.238        0.238                      0.238         0.238                  0.141        0.207                        NaN         0.237
2                      0.238        0.238                      0.238         0.238                  0.173        0.254                        NaN         0.256
3                      0.236        0.236                      0.236         0.236                  0.172        0.253                        NaN         0.251
4                      0.288        0.288                      0.288         0.288                  0.178        0.262                        NaN         0.266
5                      0.227        0.227                      0.227         0.227                  0.181        0.265                        NaN         0.236
6                      0.315        0.315                      0.315         0.315                  0.189        0.277                        NaN         0.286
7                      0.322        0.322                      0.322         0.322                  0.213        0.308                        NaN         0.322
8                      0.246        0.246                      0.246         0.246                  0.188        0.276                        NaN         0.258
9                      0.137        0.137                      0.137         0.137                  0.154        0.227                      0.159         0.207
10                     0.176        0.176                      0.176         0.176                  0.158        0.233                      0.154         0.196
11                     0.181        0.181                      0.181         0.181                  0.160        0.236                      0.177         0.193
12                     0.335        0.335                      0.335         0.335                  0.192        0.281                      0.286         0.269

The raw LogReg predictions follow the **shape** of the season but **compress it toward the middle**. They over-predict in calm Sep–Oct and under-predict December's 33.5%. The rolling-Platt column shows the correction tracking each month.

### B.2.3 Ablation: exploratory test differences

In [26]:
abl = pd.read_csv(R / "ablation.csv")
display(abl[["model", "calibration", "features", "test_pr_auc", "test_roc_auc"]].round(4))
steps[["model", "calibration", "step", "test_diff", "test_diff_ci_lo", "test_diff_ci_hi"]].round(4)

,model,calibration,features,test_pr_auc,test_roc_auc
0,Prior (base rate),none,(baseline),0.2583,0.5000
1,Route history,none (smoothed rate),(baseline),0.3534,0.6208
2,LogReg,raw,schedule + ids,0.3449,0.6124
3,LogReg,raw,+ congestion,0.3516,0.6141
4,LogReg,raw,+ history_entity,0.3844,0.6427
5,LogReg,raw,+ history_day,0.3934,0.6532
6,LogReg,raw,history only,0.3902,0.6565
7,XGBoost,raw,schedule + ids,0.3466,0.6213
8,XGBoost,raw,+ congestion,0.3456,0.6185
9,XGBoost,raw,+ history_entity,0.3939,0.6527


,model,calibration,step,test_diff,test_diff_ci_lo,test_diff_ci_hi
0,LogReg,raw,+ congestion,0.0068,-0.0098,0.0268
1,LogReg,raw,+ history_entity,0.0328,0.0158,0.0496
2,LogReg,raw,+ history_day,0.0090,-0.0094,0.0264
3,XGBoost,raw,+ congestion,-0.0010,-0.0138,0.0121
4,XGBoost,raw,+ history_entity,0.0484,0.0262,0.0698
5,XGBoost,raw,+ history_day,0.0135,-0.0201,0.0473


LogReg test AP differences are: congestion +0.0068 [−0.0098, +0.0268], entity history +0.0328 [+0.0158, +0.0496], and day-level history +0.0090 [−0.0094, +0.0264]. XGBoost differences are −0.0010 [−0.0138, +0.0121], +0.0484 [+0.0262, +0.0698], and +0.0135 [−0.0201, +0.0473], respectively. Only the entity-history intervals exclude zero in these exploratory comparisons. These results do not establish causal contributions. History-only LogReg development CV AP is **0.4480**, below the full feature set's **0.4714**.

### B.2.4 Target sensitivity: counting cancelled and diverted flights as delayed

In [27]:
sens = pd.read_csv(R / "target_sensitivity.csv")
sens.round(4)

,target,model,calibration,n_test,base_rate,pr_auc,roc_auc,brier,ece,top10_precision,top10_recall,top10_lift,xgb_minus_lr_pr_auc_ci
0,arr_delay>=15 (main),Prior (base rate),none,53991,0.2583,0.2583,0.5000,0.1916,0.0067,0.2430,0.0946,0.9408,NaN
1,arr_delay>=15 (main),Route history,none (smoothed rate),53991,0.2583,0.3534,0.6208,0.1869,0.0373,0.3581,0.1394,1.3865,NaN
2,arr_delay>=15 (main),LogReg (raw),raw,53991,0.2583,0.3935,0.6531,0.1812,0.0195,0.4475,0.1742,1.7324,"[-0.010, +0.037]"
3,arr_delay>=15 (main),XGBoost (raw),raw,53991,0.2583,0.4049,0.6624,0.1797,0.0263,0.4368,0.1700,1.6911,"[-0.010, +0.037]"
4,arr_delay>=15 OR cancelled/diverted,Prior (base rate),none,55403,0.2772,0.2772,0.5000,0.2005,0.0107,0.2622,0.0951,0.9459,NaN
5,arr_delay>=15 OR cancelled/diverted,Route history,none (smoothed rate),55403,0.2772,0.3739,0.6215,0.1958,0.0472,0.3768,0.1366,1.3593,NaN
6,arr_delay>=15 OR cancelled/diverted,LogReg (raw),raw,55403,0.2772,0.4100,0.6491,0.1902,0.0267,0.4608,0.1671,1.6625,"[-0.014, +0.045]"
7,arr_delay>=15 OR cancelled/diverted,XGBoost (raw),raw,55403,0.2772,0.4217,0.6594,0.1880,0.0252,0.4538,0.1645,1.6372,"[-0.014, +0.045]"


A tail number is missing only for cancelled flights, so with cancellations counted as delayed the tail-number history features would reveal the label. Both targets are refitted on the same 41 features without the two tail-history features (an earlier version kept all tail features and reported AP 0.446 / 0.455).

Counting cancellation/diversion proxies as delayed raises the test base rate from 25.8% to 27.7%. In that sensitivity run, raw LogReg AP is 0.410 and raw XGBoost AP is 0.422; both exceed the reference scores. Their test AP-difference interval is [−0.014, +0.045], which does not establish a difference or equivalence. Top-10% lifts are 1.66 for LogReg, 1.64 for XGBoost, and 0.95 for the prior. This is one alternative target check, not proof that findings are independent of all missing-outcome definitions.

### B.2.5 Covariate shift: PSI and adversarial validation

In [28]:
display(psi.head(12).round(3))
figs["psi_top15"]

,feature,group,psi_calib,psi_test
0,days_to_holiday,schedule,0.261,4.769
1,hist_logn_origin_7d,history_entity,2.269,1.265
2,hist_rate_network_7d,history_day,5.147,1.058
3,hist_rate_origin_7d,history_entity,3.433,0.755
4,sched_origin_day,congestion,2.056,0.610
5,hist_rate_carrier_14d,history_entity,2.631,0.383
6,hist_rate_dest_14d,history_entity,2.143,0.292
7,hist_logn_carrier_14d,history_entity,0.472,0.289
8,hist_cancel_network_1d,history_day,1.229,0.222
9,hist_rate_carrier_7d,history_entity,1.699,0.209


<Figure size 550x500 with 1 Axes>

In [29]:
pd.read_csv(R / "adversarial_validation.csv").round(3)

,comparison,features,n_features,adversarial_auc,top_features
0,train vs calib,all features,43,1.000,"hist_rate_network_7d, hist_rate_origin_7d, hist_rate_dest_14d, hist_rate_carrier_14d"
1,train vs calib,categorical,3,0.562,"dest, carrier, origin"
2,train vs calib,schedule,10,0.785,"days_to_holiday, dow_cos, dow_sin, arr_tod_sin"
3,train vs calib,congestion,5,0.984,"sched_origin_day, sched_carrier_origin_day, sched_origin_3h, sched_origin_hour"
4,train vs calib,history_entity,18,1.000,"hist_rate_origin_7d, hist_rate_dest_14d, hist_rate_carrier_14d, hist_logn_origin_7d"
5,train vs calib,history_day,7,1.000,"hist_rate_network_7d, hist_rate_network_1d, hist_logn_origin_1d, hist_cancel_network_1d"
6,train vs test,all features,43,1.000,"days_to_holiday, hist_rate_network_7d, hist_logn_origin_7d, origin"
7,train vs test,categorical,3,0.562,"dest, carrier, origin"
8,train vs test,schedule,10,0.921,"days_to_holiday, is_weekend, dow_cos, dow_sin"
9,train vs test,congestion,5,0.964,"sched_origin_day, sched_carrier_origin_day, sched_origin_3h, sched_dest_day"


* **PSI:** the largest test marginal shift is `days_to_holiday` (about 4.8); trailing history distributions also differ across periods. A threshold of 0.25 is a heuristic, not a significance test.
* **Adversarial validation:** the existing row-split classifier separates sampled training and later-period rows with high AUC, including repeated daily history values. Its family scores describe this row-level diagnostic, not validated separation on unseen days.
* PSI describes marginal feature-distribution changes; the row-split adversarial classifier describes separability of sampled rows from different periods. Daily feature values are repeated across flights, so near-perfect separability should not be interpreted as verified generalisation to unseen days. The diagnostics are consistent with distribution change, but do not establish that it caused the calibration errors or that the delay models memorised dates. The underlying diagnostic implementation is unchanged.

### B.2.6 XGBoost gain (top 20)

In [30]:
figs["xgb_gain_top20"]

<Figure size 550x640 with 1 Axes>

## Findings

1. **Availability is enforced:** history, fitting, CV selection, calibration and threshold selection use the chosen D-1 clock. Clock scenarios change several deadlines together; their score difference does not isolate a feature-leakage effect.
2. **Final model: LogReg (static Platt, rule-selected)** by development CV AP **0.4714 versus fixed-tree XGBoost 0.4500**. The test comparison does not establish a difference or equivalence. Interactions remain a later-phase hypothesis.
3. **Feature-family checks:** each added family has positive mean LogReg CV gain at the selected settings. Month features lower mean CV AP for both models.
4. **Calibration:** static LogReg test ECE **0.082**, December **0.143**. Pre-specified rolling Platt gives test ECE **0.027** for LogReg and **0.026** for XGBoost. These diagnostics do not identify the cause of the errors.
5. **Existing candidate-ranking analysis:** top-10% LogReg precision **0.448**, recall **0.174**, lift **1.73**. Ranking quality does not establish delay prevention or savings, and daily-share selection does not protect precision/recall from shift.
6. **Target sensitivity:** learned models remain above references in the tested disruption-target variant; this is a bounded sensitivity check, not invariance to every outcome definition.

## Limitations and assumptions

* **One year and three NYC origins:** cross-year and cross-airport generalisation is untested.
* **No weather forecasts:** observations from tomorrow would be unavailable at tonight's clock. Archived forecasts could be added in a later experiment; their contribution has not been measured here.
* **Partial aircraft rotation:** only NYC departures are recorded.
* **Availability assumptions:** timetable/tail assignment at 20:00 D-1, cancellation known at scheduled departure, gate-arrival outcome timing, and a fixed time-zone/DST table.
* **Development exposure:** test results were seen repeatedly (§0), and selected CV configurations are evaluated on the same development folds. Small differences are exploratory.
* **Diagnostics:** equal-mass ECE now preserves tied probabilities and is invariant to label/score row permutations. It remains binning-dependent. Row-split adversarial validation retains its documented limitation; that implementation is unchanged.